# Scientific Extraction Pipeline — Evaluation

Evaluates the output of `RAGPipeline.query_facts()`.  
Reads all `outputs/*.facts.json` files; falls back to synthetic mock data if none exist.

**Sections**
1. Setup & data loading
2. Flatten facts → DataFrame
3. Basic statistics
4. Validation metrics
5. Analysis (satellites / methods / metrics / study areas)
6. Cross-analysis (satellite → method, method → metric)
7. Error analysis (10 random facts + evidence)
8. Visualisations

## 1. Setup & data loading

In [ ]:
import json
import random
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

OUTPUTS_DIR = Path("../outputs")
FACT_FILES  = sorted(OUTPUTS_DIR.glob("*.facts.json"))
print(f"Found {len(FACT_FILES)} facts.json file(s): {[f.name for f in FACT_FILES]}")

In [ ]:
# ── Mock data (used when no real output files exist) ─────────────────────────
MOCK_DATA = [
    {
        "paper_id": "paper_001",
        "title": "Flood mapping with Sentinel-1 SAR and U-Net",
        "facts": [
            {"id": "f1", "paper_id": "paper_001", "fact_type": "data_source",
             "satellite": {"name": "Sentinel-1", "sensor_type": "SAR"},
             "evidence": [{"text": "Sentinel-1 C-band SAR data were acquired",
                           "section": "abstract+methods", "field": "Satellite_Names",
                           "source": "regex"}],
             "related_fact_ids": ["f3", "f5"]},
            {"id": "f2", "paper_id": "paper_001", "fact_type": "data_source",
             "satellite": {"name": "Sentinel-2", "sensor_type": "Optical"},
             "evidence": [{"text": "Sentinel-2 optical imagery used for validation",
                           "section": "abstract+methods", "field": "Satellite_Names",
                           "source": "regex"}],
             "related_fact_ids": ["f5"]},
            {"id": "f3", "paper_id": "paper_001", "fact_type": "method",
             "method": {"name": "U-Net", "category": "DL"},
             "evidence": [{"text": "We trained a U-Net architecture on 512×512 patches",
                           "section": "methods", "field": "Methods", "source": "regex"}],
             "related_fact_ids": ["f1", "f5"]},
            {"id": "f4", "paper_id": "paper_001", "fact_type": "method",
             "method": {"name": "Thresholding", "category": "thresholding"},
             "evidence": [{"text": "Otsu thresholding applied as baseline",
                           "section": "methods", "field": "Methods", "source": "regex"}],
             "related_fact_ids": ["f1"]},
            {"id": "f5", "paper_id": "paper_001", "fact_type": "result",
             "metric": {"type": "F1", "value": 0.91, "unit": None},
             "evidence": [{"text": "achieved F1 of 0.91 on the test set",
                           "section": "results", "field": "F1", "source": "regex"}],
             "related_fact_ids": ["f3"]},
            {"id": "f6", "paper_id": "paper_001", "fact_type": "result",
             "metric": {"type": "IoU", "value": 0.84, "unit": None},
             "evidence": [{"text": "IoU = 0.84 (flooded class)",
                           "section": "results", "field": "IoU", "source": "regex"}],
             "related_fact_ids": ["f3"]},
            {"id": "f7", "paper_id": "paper_001", "fact_type": "study_area",
             "study_area": {"country": "Ukraine", "region": "Zakarpattia",
                            "river_basin": "Tisza river basin"},
             "evidence": [{"text": "the Tisza River basin in Zakarpattia, Ukraine",
                           "section": "abstract+introduction", "field": "Country",
                           "source": "regex"}],
             "related_fact_ids": []},
            {"id": "f8", "paper_id": "paper_001", "fact_type": "task",
             "task": "flood mapping",
             "evidence": [{"text": "flood extent mapping using SAR imagery",
                           "section": "abstract", "field": "Task", "source": "rule"}],
             "related_fact_ids": ["f3", "f4", "f1", "f2"]}
        ],
        "rejected_facts": [
            {"fact": {"fact_type": "result", "metric": {"type": "OA", "value": 0.88}},
             "reasons": ["metric 'OA' evidence is from 'abstract', expected results section"]}
        ],
        "fallback_used": False
    },
    {
        "paper_id": "paper_002",
        "title": "HEC-RAS hydraulic simulation of Prut River flooding",
        "facts": [
            {"id": "f9", "paper_id": "paper_002", "fact_type": "method",
             "method": {"name": "HEC-RAS", "category": "hydraulic_model"},
             "evidence": [{"text": "HEC-RAS 2D unsteady flow simulation",
                           "section": "methods", "field": "Methods", "source": "regex"}],
             "related_fact_ids": ["f12"]},
            {"id": "f10", "paper_id": "paper_002", "fact_type": "method",
             "method": {"name": "DEM validation", "category": "DEM_validation"},
             "evidence": [{"text": "DEM accuracy assessed against GPS checkpoints",
                           "section": "methods", "field": "Methods", "source": "regex"}],
             "related_fact_ids": ["f12"]},
            {"id": "f11", "paper_id": "paper_002", "fact_type": "result",
             "metric": {"type": "RMSE", "value": 0.45, "unit": "m"},
             "evidence": [{"text": "RMSE was 0.45 m for water depth prediction",
                           "section": "results", "field": "RMSE", "source": "regex"}],
             "related_fact_ids": ["f9"]},
            {"id": "f12", "paper_id": "paper_002", "fact_type": "study_area",
             "study_area": {"country": "Ukraine", "region": "Bukovyna",
                            "river_basin": "Prut river basin"},
             "evidence": [{"text": "Prut River basin in Chernivtsi (Bukovyna) region",
                           "section": "abstract+introduction", "field": "Country",
                           "source": "regex"}],
             "related_fact_ids": []},
            {"id": "f13", "paper_id": "paper_002", "fact_type": "task",
             "task": "hydraulic simulation",
             "evidence": [{"text": "hydraulic simulation of floodplain inundation",
                           "section": "abstract", "field": "Task", "source": "rule"}],
             "related_fact_ids": ["f9", "f10"]}
        ],
        "rejected_facts": [],
        "fallback_used": False
    },
    {
        "paper_id": "paper_003",
        "title": "ICESat-2 DEM validation over Carpathians",
        "facts": [
            {"id": "f14", "paper_id": "paper_003", "fact_type": "data_source",
             "satellite": {"name": "ICESat-2", "sensor_type": "LiDAR"},
             "evidence": [{"text": "ICESat-2 ATL08 photon data used for DEM assessment",
                           "section": "abstract+methods", "field": "Satellite_Names",
                           "source": "regex"}],
             "related_fact_ids": ["f15", "f17"]},
            {"id": "f15", "paper_id": "paper_003", "fact_type": "method",
             "method": {"name": "ICESat-2 validation", "category": "DEM_validation"},
             "evidence": [{"text": "ICESat-2 ATL06 land ice height product used as reference",
                           "section": "methods", "field": "Methods", "source": "regex"}],
             "related_fact_ids": ["f14", "f17"]},
            {"id": "f16", "paper_id": "paper_003", "fact_type": "result",
             "metric": {"type": "RMSE", "value": 1.2, "unit": "m"},
             "evidence": [{"text": "vertical RMSE of 1.2 m against ICESat-2 checkpoints",
                           "section": "results", "field": "RMSE", "source": "regex"}],
             "related_fact_ids": ["f15"]},
            {"id": "f17", "paper_id": "paper_003", "fact_type": "study_area",
             "study_area": {"country": "Ukraine", "region": "Carpathians",
                            "river_basin": None},
             "evidence": [{"text": "Ukrainian Carpathians, elevation 500–2000 m",
                           "section": "abstract+introduction", "field": "Country",
                           "source": "regex"}],
             "related_fact_ids": []},
            {"id": "f18", "paper_id": "paper_003", "fact_type": "task",
             "task": "DEM accuracy assessment",
             "evidence": [{"text": "DEM accuracy assessment using LiDAR-derived reference",
                           "section": "abstract", "field": "Task", "source": "rule"}],
             "related_fact_ids": ["f15", "f14"]},
            {"id": "f19", "paper_id": "paper_003", "fact_type": "system_property",
             "value": "near_real_time",
             "evidence": [{"text": "near-real-time elevation monitoring from orbit",
                           "section": "abstract+methods", "field": "Near_Real_Time",
                           "source": "regex"}],
             "related_fact_ids": ["f14", "f15"]}
        ],
        "rejected_facts": [
            {"fact": {"fact_type": "data_source", "satellite": {"name": "UNKNOWN_SAT"}},
             "reasons": ["satellite 'UNKNOWN_SAT' is not in the known list and no evidence text found"]}
        ],
        "fallback_used": False
    }
]

# ── Load real data or fall back to mock ──────────────────────────────────────
results = []
if FACT_FILES:
    for fp in FACT_FILES:
        with open(fp) as fh:
            data = json.load(fh)
        # Accept either a single result dict or a list
        if isinstance(data, list):
            results.extend(data)
        else:
            results.append(data)
    print(f"Loaded {len(results)} paper result(s) from disk.")
else:
    results = MOCK_DATA
    print("No facts.json files found — using mock data (3 papers).")

## 2. Flatten facts → DataFrame

In [ ]:
def flatten_facts(results: list) -> pd.DataFrame:
    rows = []
    for r in results:
        paper_id = r.get("paper_id", "")
        title    = r.get("title", "")
        for f in r.get("facts", []):
            ev      = f.get("evidence", [])
            sat     = f.get("satellite") or {}
            method  = f.get("method") or {}
            metric  = f.get("metric") or {}
            area    = f.get("study_area") or {}

            ev_sections = list({e.get("section", "") for e in ev if e.get("section")})

            rows.append({
                "paper_id":           paper_id,
                "title":              title,
                "fact_id":            f.get("id", ""),
                "fact_type":          f.get("fact_type", ""),
                "satellite_name":     sat.get("name"),
                "sensor_type":        sat.get("sensor_type"),
                "method_name":        method.get("name"),
                "method_category":    method.get("category"),
                "metric_type":        metric.get("type"),
                "metric_value":       metric.get("value"),
                "metric_unit":        metric.get("unit"),
                "country":            area.get("country"),
                "region":             area.get("region"),
                "river_basin":        area.get("river_basin"),
                "task":               f.get("task"),
                "system_property":    f.get("value"),
                "evidence_count":     len(ev),
                "has_evidence":       len(ev) > 0,
                "evidence_sections":  ev_sections,
                "related_fact_count": len(f.get("related_fact_ids", [])),
            })
    return pd.DataFrame(rows)


df = flatten_facts(results)
print(f"DataFrame shape: {df.shape}")
df.head()

## 3. Basic statistics

In [ ]:
n_papers   = df["paper_id"].nunique()
n_facts    = len(df)
avg_facts  = n_facts / n_papers if n_papers else 0

type_counts = df["fact_type"].value_counts()
fallback_n  = sum(1 for r in results if r.get("fallback_used"))
total_rejected = sum(len(r.get("rejected_facts", [])) for r in results)

print(f"Papers processed  : {n_papers}")
print(f"Total accepted facts: {n_facts}")
print(f"Avg facts / paper : {avg_facts:.1f}")
print(f"Fallback used     : {fallback_n} paper(s)")
print(f"Rejected facts    : {total_rejected}")
print("\nFact type distribution:")
print(type_counts.to_string())

## 4. Validation metrics

In [ ]:
pct_with_evidence = df["has_evidence"].mean() * 100

# ── % result facts whose evidence section is 'results' ────────────────────────
result_df = df[df["fact_type"] == "result"].copy()
if len(result_df):
    result_df["from_results_section"] = result_df["evidence_sections"].apply(
        lambda secs: "results" in secs
    )
    pct_results_from_results = result_df["from_results_section"].mean() * 100
else:
    pct_results_from_results = float("nan")

# ── % method facts whose evidence is from methods section ─────────────────────
method_df = df[df["fact_type"] == "method"].copy()
if len(method_df):
    method_df["from_methods_section"] = method_df["evidence_sections"].apply(
        lambda secs: bool({"methods", "abstract+methods"} & set(secs))
    )
    pct_methods_from_methods = method_df["from_methods_section"].mean() * 100
else:
    pct_methods_from_methods = float("nan")

# ── Cross-links ───────────────────────────────────────────────────────────────
pct_linked = (df["related_fact_count"] > 0).mean() * 100

print(f"Facts with evidence              : {pct_with_evidence:.1f}%")
print(f"Result facts from 'results'      : {pct_results_from_results:.1f}%")
print(f"Method facts from 'methods'      : {pct_methods_from_methods:.1f}%")
print(f"Facts with cross-links           : {pct_linked:.1f}%")

## 5. Analysis

In [ ]:
# ── 5a. Top satellites ────────────────────────────────────────────────────────
sat_counts = (
    df[df["fact_type"] == "data_source"]["satellite_name"]
    .value_counts()
    .head(15)
)
print("Top satellites:")
print(sat_counts.to_string() if len(sat_counts) else "  (none found)")

# ── 5b. Sensor-type breakdown ─────────────────────────────────────────────────
sensor_counts = (
    df[df["fact_type"] == "data_source"]["sensor_type"]
    .value_counts()
)
print("\nSensor-type distribution:")
print(sensor_counts.to_string() if len(sensor_counts) else "  (none found)")

In [ ]:
# ── 5c. Top methods ───────────────────────────────────────────────────────────
meth_counts = (
    df[df["fact_type"] == "method"]["method_name"]
    .value_counts()
    .head(15)
)
print("Top methods:")
print(meth_counts.to_string() if len(meth_counts) else "  (none found)")

cat_counts = (
    df[df["fact_type"] == "method"]["method_category"]
    .value_counts()
)
print("\nMethod-category distribution:")
print(cat_counts.to_string() if len(cat_counts) else "  (none found)")

In [ ]:
# ── 5d. Metric distributions ──────────────────────────────────────────────────
metric_df = df[df["fact_type"] == "result"].copy()
print("Metric type distribution:")
print(metric_df["metric_type"].value_counts().to_string() if len(metric_df) else "  (none found)")

if len(metric_df):
    print("\nMetric value summary (per type):")
    display(
        metric_df.groupby("metric_type")["metric_value"]
        .agg(["count", "mean", "std", "min", "max"])
        .round(3)
    )

In [ ]:
# ── 5e. Study areas ───────────────────────────────────────────────────────────
area_df = df[df["fact_type"] == "study_area"]
print("Countries:")
print(area_df["country"].value_counts().to_string() if len(area_df) else "  (none found)")
print("\nRegions:")
print(area_df["region"].value_counts().to_string() if len(area_df) else "  (none found)")
print("\nRiver basins:")
print(area_df["river_basin"].dropna().value_counts().to_string() if len(area_df) else "  (none found)")

In [ ]:
# ── 5f. Task distribution ─────────────────────────────────────────────────────
task_counts = df[df["fact_type"] == "task"]["task"].value_counts()
print("Task distribution:")
print(task_counts.to_string() if len(task_counts) else "  (none found)")

## 6. Cross-analysis

In [ ]:
# ── Build fact lookup for cross-analysis ──────────────────────────────────────
fact_lookup: dict[str, dict] = {}
for r in results:
    for f in r.get("facts", []):
        fact_lookup[f["id"]] = f

# ── 6a. Satellite → Method cross-frequency ────────────────────────────────────
sat_meth_pairs = []
for f in fact_lookup.values():
    if f["fact_type"] == "method":
        mname = (f.get("method") or {}).get("name")
        if not mname:
            continue
        for related_id in f.get("related_fact_ids", []):
            related = fact_lookup.get(related_id, {})
            if related.get("fact_type") == "data_source":
                sname = (related.get("satellite") or {}).get("name")
                if sname:
                    sat_meth_pairs.append({"satellite": sname, "method": mname})

if sat_meth_pairs:
    sm_df = pd.DataFrame(sat_meth_pairs)
    sm_pivot = sm_df.pivot_table(
        index="method", columns="satellite", aggfunc="size", fill_value=0
    )
    print("Satellite → Method co-occurrence matrix:")
    display(sm_pivot)
else:
    sm_pivot = None
    print("No satellite–method cross-links found.")

In [ ]:
# ── 6b. Method → Metric cross-frequency ──────────────────────────────────────
meth_metric_pairs = []
for f in fact_lookup.values():
    if f["fact_type"] == "result":
        mtype = (f.get("metric") or {}).get("type")
        if not mtype:
            continue
        for related_id in f.get("related_fact_ids", []):
            related = fact_lookup.get(related_id, {})
            if related.get("fact_type") == "method":
                mname = (related.get("method") or {}).get("name")
                if mname:
                    meth_metric_pairs.append({"method": mname, "metric": mtype})

if meth_metric_pairs:
    mm_df = pd.DataFrame(meth_metric_pairs)
    mm_pivot = mm_df.pivot_table(
        index="metric", columns="method", aggfunc="size", fill_value=0
    )
    print("Method → Metric co-occurrence matrix:")
    display(mm_pivot)
else:
    mm_pivot = None
    print("No method–metric cross-links found.")

## 7. Error analysis — 10 random facts with evidence

In [ ]:
all_facts_flat = [f for r in results for f in r.get("facts", [])]
sample = random.sample(all_facts_flat, min(10, len(all_facts_flat)))

for i, f in enumerate(sample, 1):
    sat    = (f.get("satellite") or {}).get("name", "")
    method = (f.get("method")    or {}).get("name", "")
    metric = f.get("metric")
    area   = f.get("study_area") or {}
    label  = sat or method or (f"{metric['type']}={metric['value']}" if metric else "") \
             or area.get("country", "") or f.get("task") or f.get("value", "")

    print(f"[{i:02d}] {f['fact_type']:16s}  {label}  (paper: {f['paper_id']})")
    for ev in f.get("evidence", []):
        print(f"       [{ev.get('section','?')} / {ev.get('source','?')}]  {ev.get('text','')[:120]}")
    if not f.get("evidence"):
        print("       *** NO EVIDENCE ***")
    print()

In [ ]:
# ── Rejected facts ────────────────────────────────────────────────────────────
all_rejected = [
    {"paper_id": r["paper_id"], **rej}
    for r in results
    for rej in r.get("rejected_facts", [])
]

if all_rejected:
    print(f"Rejected facts: {len(all_rejected)}")
    for rej in all_rejected[:10]:
        ftype = (rej.get("fact") or {}).get("fact_type", "?")
        reasons = rej.get("reasons", [])
        print(f"  [{rej['paper_id']}] {ftype}: {'; '.join(reasons)}")
else:
    print("No rejected facts found.")

## 8. Visualisations

In [ ]:
# ── 8a. Fact-type distribution (bar chart) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
type_counts_sorted = df["fact_type"].value_counts().sort_values(ascending=True)
bars = ax.barh(type_counts_sorted.index, type_counts_sorted.values, color=sns.color_palette("muted", len(type_counts_sorted)))
ax.bar_label(bars, padding=4)
ax.set_xlabel("Count")
ax.set_title("Fact-type distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ── 8b. Top satellites (bar chart) ───────────────────────────────────────────
if len(sat_counts):
    fig, ax = plt.subplots(figsize=(9, 4))
    sat_counts_sorted = sat_counts.sort_values()
    bars = ax.barh(sat_counts_sorted.index, sat_counts_sorted.values,
                   color=sns.color_palette("Set2", len(sat_counts_sorted)))
    ax.bar_label(bars, padding=4)
    ax.set_xlabel("Count")
    ax.set_title("Most common satellites")
    plt.tight_layout()
    plt.show()
else:
    print("No satellite facts to plot.")

In [ ]:
# ── 8c. Top methods (bar chart) ───────────────────────────────────────────────
if len(meth_counts):
    fig, ax = plt.subplots(figsize=(9, 5))
    mc_sorted = meth_counts.sort_values()
    bars = ax.barh(mc_sorted.index, mc_sorted.values,
                   color=sns.color_palette("Set3", len(mc_sorted)))
    ax.bar_label(bars, padding=4)
    ax.set_xlabel("Count")
    ax.set_title("Most common methods")
    plt.tight_layout()
    plt.show()
else:
    print("No method facts to plot.")

In [ ]:
# ── 8d. Metric histograms (one subplot per metric type) ───────────────────────
if len(metric_df):
    metric_types = metric_df["metric_type"].dropna().unique()
    n_cols = min(3, len(metric_types))
    n_rows = (len(metric_types) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)

    for idx, mtype in enumerate(metric_types):
        ax = axes[idx // n_cols][idx % n_cols]
        vals = metric_df[metric_df["metric_type"] == mtype]["metric_value"].dropna()
        ax.hist(vals, bins=max(5, min(20, len(vals))), color="steelblue", edgecolor="white")
        ax.set_title(f"{mtype} (n={len(vals)})")
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")

    # Hide any spare axes
    for idx in range(len(metric_types), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    plt.suptitle("Metric value distributions", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No result facts to plot.")

In [ ]:
# ── 8e. Satellite × Method heatmap ───────────────────────────────────────────
if sm_pivot is not None and sm_pivot.size > 0:
    fig, ax = plt.subplots(figsize=(max(6, sm_pivot.shape[1] * 1.4),
                                    max(4, sm_pivot.shape[0] * 0.8)))
    sns.heatmap(sm_pivot, annot=True, fmt="d", cmap="YlOrRd",
                linewidths=0.5, ax=ax, cbar_kws={"label": "co-occurrences"})
    ax.set_title("Satellite × Method co-occurrence")
    ax.set_xlabel("Satellite")
    ax.set_ylabel("Method")
    plt.tight_layout()
    plt.show()
else:
    print("Heatmap skipped — no satellite–method links.")

In [ ]:
# ── 8f. Method × Metric heatmap ──────────────────────────────────────────────
if mm_pivot is not None and mm_pivot.size > 0:
    fig, ax = plt.subplots(figsize=(max(6, mm_pivot.shape[1] * 1.4),
                                    max(4, mm_pivot.shape[0] * 0.8)))
    sns.heatmap(mm_pivot, annot=True, fmt="d", cmap="Blues",
                linewidths=0.5, ax=ax, cbar_kws={"label": "co-occurrences"})
    ax.set_title("Method × Metric co-occurrence")
    ax.set_xlabel("Method")
    ax.set_ylabel("Metric")
    plt.tight_layout()
    plt.show()
else:
    print("Heatmap skipped — no method–metric links.")

In [ ]:
# ── 8g. Validation scorecard (bar chart) ────────────────────────────────────
scorecard = {
    "Has evidence": pct_with_evidence,
    "Results from\n'results' section": pct_results_from_results,
    "Methods from\n'methods' section": pct_methods_from_methods,
    "Has cross-links": pct_linked,
}
# Drop NaN entries
scorecard = {k: v for k, v in scorecard.items() if not (isinstance(v, float) and np.isnan(v))}

fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ["#2ecc71" if v >= 80 else "#e67e22" if v >= 50 else "#e74c3c"
          for v in scorecard.values()]
bars = ax.barh(list(scorecard.keys()), list(scorecard.values()), color=colors)
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set_xlim(0, 115)
ax.set_xlabel("Percentage (%)")
ax.set_title("Pipeline validation scorecard")
plt.tight_layout()
plt.show()